# Forest Covertype

Dataset-specific starting notebook for the DataFrameSampler paper experiments.

Claim-specific role: larger public benchmark with continuous terrain measurements and categorical wilderness/soil context.

## Setup

Run the downloader before executing this notebook:

```bash
python experiments/download_datasets.py
```

Dataset-specific choices live in `experiments/datasets.py`; the reusable execution path lives in `experiments/workflow.py`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "experiments").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.datasets import DATASET_CONFIGS
from experiments.exploration import column_distribution_summary, plot_column_distribution_comparison, plot_pairwise_feature_comparison
from experiments.manifold_validation import summarize_manifold_validation
from experiments.mechanism_validation import summarize_decoder_calibration, summarize_mechanism_validation
from experiments.predictive import predictive_performance_report, target_column_choice
from experiments.vectorization_plan import preprocessing_plan, vectorization_plan
from experiments.workflow import dataset_profile, experiment_paths, load_dataset, notebook_environment, run_configured_dataset_experiment, working_dataframe

DATASET_NAME = "covertype"
CONFIG = DATASET_CONFIGS[DATASET_NAME]
PATHS = experiment_paths(CONFIG, root=ROOT)
notebook_environment(PATHS)

## Load Prepared Data

In [ ]:
dataframe = load_dataset(CONFIG, root=ROOT)
work = working_dataframe(dataframe, CONFIG)
profile = dataset_profile(dataframe)
dataframe.shape, dataframe.head()

## Dataset Profile

In [ ]:
display(profile)
display(target_column_choice(CONFIG, profile))

## DataFrameSampler Plan

In [ ]:
pre_plan = preprocessing_plan(CONFIG)
if not pre_plan.empty:
    display(pre_plan)

vectorization_plan(work, CONFIG)[["column", "strategy", "latent_components", "rationale"]]

## Original Data

In [ ]:
dataframe.head()

## Run Experiment

In [ ]:
result = run_configured_dataset_experiment(CONFIG, root=ROOT)
result.paths

## Generated Data

In [ ]:
generated = result.starter_run.generated
generated.head()

## Original vs Generated

In [ ]:
_ = plot_column_distribution_comparison(work, generated, title="Forest Covertype: original vs generated")
_ = plot_pairwise_feature_comparison(work, generated, target_column=CONFIG.target_column, title="Forest Covertype: pairwise comparison")

## Summaries

In [ ]:
display(result.comparison)
display(summarize_manifold_validation(result.manifold_validation))
display(summarize_mechanism_validation(result.mechanism_validation))
display(summarize_decoder_calibration(result.decoder_calibration))
display(predictive_performance_report(work, generated, target_column=CONFIG.target_column, random_state=CONFIG.random_state))